# Heterozygosity and runs of homozygosity (Ag3)

This notebook demonstrates the `Ag3` methods for analysing per-sample
heterozygosity and inferring runs of homozygosity (ROH):
`plot_heterozygosity`, `roh_hmm`, `plot_roh`, and `cohort_heterozygosity`.

Heterozygosity here means the proportion of genotype calls in a sliding
window of SNPs that are heterozygous. A long stretch of unusually low
heterozygosity ("run of homozygosity") can indicate inbreeding or a
recent shared ancestor.

In [1]:
import malariagen_data
ag3 = malariagen_data.Ag3(
    "simplecache::gs://vo_agam_release_master_us_central1",
    simplecache=dict(cache_storage="../../gcs_cache"),
    results_cache="../../results_cache",
)
ag3

/opt/homebrew/Caskroom/miniconda/base/envs/malariagen2/lib/python3.11/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


<MalariaGEN Ag3 API client>
Storage URL                           : simplecache::gs://vo_agam_release_master_us_central1
Data releases available               : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16
Results cache                         : /Users/katie.barr/malariagen-data-python/results_cache
Cohorts analysis                      : 20260120
AIM analysis                          : 20220528
Site filters analysis                 : dt_20200416
Software version                      : malariagen_data 15.8.0.post13+b769b728
Client location                       : England, United Kingdom
Data filtered to unrestricted use only: False
Data filtered to surveillance use only: False
Relevant data releases                : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16
---
Please note that data are subject to terms of use,
for more information see the Vector Observatory website https://www.malariagen.net/vobs/
or contact support@malariagen.net. For API documentation see 
https://malariagen.github.io/malariagen-data-python/v15.8.0.post13+b769b728/Ag3.html

## Selecting a sample

The single-sample methods below need one real `sample_id`. We pull it from
`sample_metadata()` for the `AG1000G-KE` sample set (a modest-sized sample
set), and use the first sample, `AK0050-C`.

In [2]:
df_samples = ag3.sample_metadata(sample_sets="AG1000G-KE")
df_samples[["sample_id", "sample_set", "country", "taxon", "year"]].head()

Load sample metadata: ⠋ (0:00:00.00)

,sample_id,sample_set,country,taxon,year
0,AK0041-C,AG1000G-KE,Kenya,arabiensis,2012
1,AK0042-C,AG1000G-KE,Kenya,arabiensis,2012
2,AK0043-C,AG1000G-KE,Kenya,arabiensis,2012
3,AK0044-C,AG1000G-KE,Kenya,arabiensis,2012
4,AK0046-C,AG1000G-KE,Kenya,arabiensis,2012


## `plot_heterozygosity`

Plots windowed heterozygosity for one or more samples over a genome region,
as an interactive Bokeh track, with a genes track underneath for context.

Parameters:
- **sample**: a sample identifier/index, or a list/tuple of them. Passing a
  list stacks one heterozygosity track per sample, all sharing the same
  x-axis, which is useful for comparing individuals.
- **region**: the genome region to plot (contig, `"contig:start-end"`
  string, or a gene/transcript ID). Larger regions take longer to compute
  and render.
- **window_size**: number of SNP sites per sliding window (default 20,000).
  Smaller windows give finer spatial resolution but noisier heterozygosity
  estimates; larger windows smooth the signal.
- **y_max**: the y-axis limit for heterozygosity (default 0.03). Change it
  to rescale the plot if a sample has unusually high or low heterozygosity.
- **circle_kwargs**: dict passed through to Bokeh's `circle()` marker (e.g.
  to change point size or colour).
- **site_mask**: which site-filter mask to apply (e.g. `"gamb_colu"`),
  restricting to sites passing quality filters for that taxon group. `None`
  disables filtering.
- **sample_set**: explicitly disambiguate which sample set a `sample_id`
  belongs to, if it is not unique across the resource.
- **sizing_mode**: Bokeh responsive-sizing behaviour for the figure.
- **width**: plot width in pixels; `None` lets `sizing_mode` control it.
- **track_height**: height in pixels of each heterozygosity track.
- **genes_height**: height in pixels of the genes track panel.
- **show**: if `True` (default), displays the plot; if `False`, returns the
  Bokeh figure object instead (useful for combining figures programmatically).
- **output_backend**: Bokeh rendering backend (`"webgl"`, `"canvas"`, or
  `"svg"`); webgl is fastest for plots with many points.
- **chunks** / **inline_array**: control how the underlying Dask/Zarr arrays
  are chunked and loaded; rarely need changing for typical use.
- **gene_labels**: a mapping of gene ID to custom label text shown in the
  genes track.
- **gene_labelset**: an explicit Bokeh `LabelSet` to use for the genes
  track, overriding automatic labelling.

We use a small gene region (`AGAP004707`, the *Vgsc* gene) rather than a
whole chromosome arm, and a small `window_size`, to keep computation and
rendering fast for this example.

In [3]:
ag3.plot_heterozygosity(
    sample="AK0050-C",
    region="AGAP004707",
    site_mask="gamb_colu",
    window_size=1_000,
)

Load genome features: ⠋ (0:00:00.00)

Load genome features: ⠙ (0:00:00.09)

Load genome features: ⠹ (0:00:00.19)

Load genome features: ⠸ (0:00:00.33)

Load genome features: ⠼ (0:00:00.42)

...

Load genome features: ⠙ (0:00:00.08)

Load genome features: ⠹ (0:00:00.17)

Load genome features: ⠸ (0:00:00.29)

Load genome features: ⠼ (0:00:00.46)

GridPlot(id='p1100', ...)

## `roh_hmm`

Infers runs of homozygosity (ROH) for a single sample over a genome region,
using a Hidden Markov Model (HMM) fitted to windowed heterozygous-site
counts. Returns a DataFrame with one row per inferred ROH block.

Parameters:
- **sample**: a single sample identifier or index (unlike `plot_heterozygosity`,
  only one sample is allowed here).
- **region**: genome region to analyse.
- **window_size**: number of SNP sites per window used to compute
  heterozygous-site counts that feed the HMM. Larger windows give smoother
  input counts but coarser ROH boundaries.
- **site_mask**: site-filter mask to apply before counting heterozygotes.
- **sample_set**: disambiguate the sample set for a non-unique `sample_id`.
- **phet_roh**: the probability of observing a heterozygous call while
  inside a ROH state (default 0.001). Lower values make the model stricter
  about calling ROH (it expects almost no heterozygotes within a run).
- **phet_nonroh**: one or more probabilities of a heterozygous call while
  outside a ROH (default `(0.003, 0.01)`). Supplying more than one value
  gives the HMM multiple "non-ROH" states of differing heterozygosity,
  letting it model heterogeneous background heterozygosity.
- **transition**: the probability of moving between hidden states (default
  0.001). A higher value makes the HMM switch between ROH/non-ROH more
  readily, producing more (and shorter) runs; a lower value favours fewer,
  longer runs. Larger `window_size` may call for a larger `transition`.
- **chunks** / **inline_array**: Dask/Zarr loading controls, as above.

**Diagram opportunity:** a schematic showing a chromosome track with a
simulated genotype sequence, the windowed heterozygosity signal derived
from it, and the HMM's inferred hidden-state path (ROH vs non-ROH) aligned
underneath — to make concrete what "probability of a het call in/out of a
ROH state" and "transition probability" actually control. Would fit here,
before or after this code cell in `12_heterozygosity_analysis.ipynb`.

Because ROH tends to occur as long, sparse stretches, a very small region
like a single gene rarely contains a complete run, so here we use the
larger `"3R"` chromosome arm (still a single sample, so this remains fast)
to get an interesting, non-empty result.

In [4]:
df_roh = ag3.roh_hmm(
    sample="AK0050-C",
    region="3R",
    site_mask="gamb_colu",
    window_size=20_000,
)
df_roh.head()

,roh_start,roh_stop,roh_length,roh_is_marginal,sample_id,contig
0,180,1690670,1690490,True,AK0050-C,3R
1,1734757,2003686,268929,False,AK0050-C,3R
2,11300062,12567018,1266956,False,AK0050-C,3R
3,12593955,14917237,2323282,False,AK0050-C,3R
4,18307285,18590424,283139,False,AK0050-C,3R


## `plot_roh`

Combines the heterozygosity track, the inferred ROH track, and a genes
track into one linked figure for a single sample and region. Internally
this calls the same windowed-heterozygosity computation as
`plot_heterozygosity`/`sample_count_het`, and the same ROH inference as
`roh_hmm`, so the parameters are the union of those two methods' parameters,
plus a few plot-layout options:

- **sample**, **region**, **window_size**, **site_mask**, **sample_set**:
  as in `plot_heterozygosity`/`roh_hmm`.
- **phet_roh**, **phet_nonroh**, **transition**: HMM parameters, as in
  `roh_hmm`.
- **y_max**: y-axis limit for the heterozygosity sub-track.
- **sizing_mode**, **width**: Bokeh sizing controls for the combined figure.
- **heterozygosity_height**: height in pixels of the heterozygosity
  sub-track (distinct from `roh_height` and `genes_height`, since this
  figure stacks three tracks).
- **roh_height**: height in pixels of the ROH sub-track.
- **genes_height**: height in pixels of the genes sub-track.
- **circle_kwargs**: passed to the heterozygosity scatter markers.
- **show**, **output_backend**, **chunks**, **inline_array**, **gene_labels**,
  **gene_labelset**: as above.

We reuse the same sample/region/window as the `roh_hmm` example above, so
the ROH blocks are visible in the combined plot.

In [5]:
ag3.plot_roh(
    sample="AK0050-C",
    region="3R",
    site_mask="gamb_colu",
    window_size=20_000,
)

Access SNP calls: ⠋ (0:00:00.00)

Apply site filters: ⠋ (0:00:00.00)

Apply site filters: ⠙ (0:00:00.09)

Apply site filters: ⠹ (0:00:00.17)

Apply site filters: ⠸ (0:00:00.26)

Compute heterozygous genotypes:   0%|          | 0/351 [00:00<?, ?it/s]

Load genome features: ⠋ (0:00:00.00)

GridPlot(id='p1250', ...)

## `cohort_heterozygosity`

Computes *mean* heterozygosity per cohort (a group of samples), over a
genome region — i.e. it aggregates windowed heterozygosity across all
samples and all windows within each cohort, rather than returning a
per-window track.

Parameters:
- **region**: genome region to analyse.
- **cohorts**: either the name of a predefined cohort-grouping column in
  the sample metadata (e.g. `"admin1_year"`), or a dict mapping custom
  cohort labels to pandas sample-metadata queries (used here).
- **sample_sets**: restrict to particular sample set(s)/release(s).
- **sample_query**: an additional pandas query applied on top of each
  cohort's own query, to further restrict samples.
- **sample_query_options**: extra kwargs passed through to pandas'
  `query()`/`eval()`.
- **window_size**: SNP sites per window used internally when computing
  heterozygosity per sample, before averaging across windows and samples.
- **site_mask**: site-filter mask to apply.
- **chunks** / **inline_array**: Dask/Zarr loading controls.

**Note on runtime:** this method loads full genotype data for every sample
in every cohort over the requested region, which the package maintainers
have flagged as slow for large regions or large cohorts (e.g. a whole
chromosome arm across hundreds of samples). To keep this example fast, we
deliberately use a small region (the `AGAP004707` gene, ~73 kbp, rather
than a whole chromosome arm) and a single, modestly-sized cohort restricted
to one sample set and one taxon.

In [6]:
ag3.cohort_heterozygosity(
    region="AGAP004707",
    cohorts={"GH_coluzzii": "taxon == 'coluzzii'"},
    sample_sets="AG1000G-GH",
    window_size=1_000,
    site_mask="gamb_colu",
)

Load genome features: ⠋ (0:00:00.00)

Load sample metadata: ⠋ (0:00:00.00)

Access SNP calls: ⠋ (0:00:00.00)

Access SNP calls: ⠙ (0:00:00.09)

Apply site filters: ⠋ (0:00:00.00)

...

Compute heterozygous genotypes:   0%|          | 0/15 [00:00<?, ?it/s]

Compute heterozygous genotypes:   0%|          | 0/15 [00:00<?, ?it/s]

Compute heterozygous genotypes:   0%|          | 0/15 [00:00<?, ?it/s]

Compute heterozygous genotypes:   0%|          | 0/15 [00:00<?, ?it/s]

,cohort,n_samples,mean_heterozygosity
0,GH_coluzzii,64,0.001273
